<a href="https://colab.research.google.com/github/aishanee-sinha/Multi-Agent-Autonomous-Workforce-Assistant/blob/aishanee-codes/Email_summarizer_enron.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("wcukierski/enron-email-dataset")

print("Path to dataset files:", path)


Using Colab cache for faster access to the 'enron-email-dataset' dataset.
Path to dataset files: /kaggle/input/enron-email-dataset


In [3]:
import os
df = pd.read_csv(os.path.join(path, "emails.csv"))

In [4]:
from email import message_from_string

def parse_email_id(raw_msg):
    try:
        msg = message_from_string(raw_msg)

        #extract the headers for the message column
        msg_id = msg.get("Message-ID")
        sender = msg.get("From")
        receiver = msg.get("To")
        date = msg.get("Date")
        subject = msg.get("Subject")
        body = ""
        if msg.is_multipart():
            for part in msg.walk():
                if part.get_content_type() == "text/plain":
                    body += part.get_payload(decode=True).decode(errors="ignore")
        else:
            payload = msg.get_payload(decode=True)
            if payload:
                body = payload.decode(errors="ignore")

        return msg_id, sender, receiver, date, subject, body.strip()

    except Exception:
        return None, None, None, None, None, None

#apply parsing
parsed = df["message"].apply(parse_email_id)

df_parsed = pd.DataFrame(parsed.tolist(),
                         columns=["message_id", "from", "to", "date", "subject", "body"])

# keep original file column
df_final = pd.concat([df["file"], df_parsed], axis=1)

# Convert date
df_final["date"] = pd.to_datetime(df_final["date"], errors="coerce")

df_final.head(3)


/tmp/ipython-input-3704971510.py:38: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_final["date"] = pd.to_datetime(df_final["date"], errors="coerce")
/tmp/ipython-input-3704971510.py:38: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df_final["date"] = pd.to_datetime(df_final["date"], errors="coerce")


,file,message_id,from,to,date,subject,body
0,allen-p/_sent_mail/1.,<18782981.1075855378110.JavaMail.evans@thyme>,phillip.allen@enron.com,tim.belden@enron.com,2001-05-14 16:39:00-07:00,,Here is our forecast
1,allen-p/_sent_mail/10.,<15464986.1075855378456.JavaMail.evans@thyme>,phillip.allen@enron.com,john.lavorato@enron.com,2001-05-04 13:51:00-07:00,Re:,Traveling to have a business meeting takes the...
2,allen-p/_sent_mail/100.,<24216240.1075855687451.JavaMail.evans@thyme>,phillip.allen@enron.com,leah.arsdall@enron.com,2000-10-18 03:00:00-07:00,Re: test,test successful. way to go!!!


In [5]:
#filter one email with greater than 100 words in the body
df_long = df_final[df_final["body"].str.len() > 100]


In [6]:
from transformers import pipeline
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


AttributeError: 'DataFrame' object has no attribute 'idloc'

In [7]:
long_email = f"""{df_long.iloc[0]['body']}"""

In [8]:
print("Original email")
print(df_long.iloc[0]['body'])
summary = summarizer(long_email, max_length=150, min_length=50, do_sample=False)
print("Summarized email")
print(summary[0]['summary_text'])

Original email
Traveling to have a business meeting takes the fun out of the trip.  Especially if you have to prepare a presentation.  I would suggest holding the business plan meetings here then take a trip without any formal business meetings.  I would even try and get some honest opinions on whether a trip is even desired or necessary.

As far as the business meetings, I think it would be more productive to try and stimulate discussions across the different groups about what is working and what is not.  Too often the presenter speaks and the others are quiet just waiting for their turn.   The meetings might be better if held in a round table discussion format.  

My suggestion for where to go is Austin.  Play golf and rent a ski boat and jet ski's.  Flying somewhere takes too much time.
Summarized email
Traveling to have a business meeting takes the fun out of the trip. I would suggest holding the business plan meetings here then take a trip without any formal business meetings. Pla

In [12]:
#print original and summary email with line breaks after specific number of characters to read easily on screen without scrolling
def print_with_line_breaks(text, line_length=80):
    lines = []
    current_line = ""
    for word in text.split():
        if len(current_line) + len(word) + 1 <= line_length:
            current_line += word + " "
        else:
            lines.append(current_line.strip())
            current_line = word + " "
    lines.append(current_line.strip())
    return "\n".join(lines)
print("Original email")
print(print_with_line_breaks(df_long.iloc[0]['body']))

Original email
Traveling to have a business meeting takes the fun out of the trip. Especially
if you have to prepare a presentation. I would suggest holding the business
plan meetings here then take a trip without any formal business meetings. I
would even try and get some honest opinions on whether a trip is even desired
or necessary. As far as the business meetings, I think it would be more
productive to try and stimulate discussions across the different groups about
what is working and what is not. Too often the presenter speaks and the others
are quiet just waiting for their turn. The meetings might be better if held in
a round table discussion format. My suggestion for where to go is Austin. Play
golf and rent a ski boat and jet ski's. Flying somewhere takes too much time.


In [11]:
print("Summarized email")
print(print_with_line_breaks(summary[0]['summary_text']))

Summarized email
Traveling to have a business meeting takes the fun out of the trip. I would
suggest holding the business plan meetings here then take a trip without any
formal business meetings. Play golf and rent a ski boat and jet ski's. Flying
somewhere takes too much time.
